#Experimentos

In [1]:
!pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 9.2 MB/s eta 0:00:00


##Existência

In [2]:
from rdflib import Graph, Namespace, RDF, URIRef

noo = Namespace("http://noosphera.org/ontology#")

grafo = Graph()

O1 = noo["O1"]

grafo.add((O1, RDF.type, noo.Object))

print("Número de triplas:", len(grafo))

for s, p, o in grafo:
    print(s, p, o)

Número de triplas: 1
http://noosphera.org/ontology#O1 http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://noosphera.org/ontology#Object


#Estrutura

In [3]:
from rdflib import Graph, Namespace, RDF

noo = Namespace("http://noosphera.org/ontology#")

grafo = Graph()

O1 = noo["O1"]
O2 = noo["O2"]

grafo.add((O1, RDF.type, noo.Object))
grafo.add((O2, RDF.type, noo.Object))

relacao = noo["relatesTo"]

grafo.add((O1, relacao, O2))

print("Número de triplas:", len(grafo))

for s, p, o in grafo:
    print(f"{s} -> {p} -> {o}")

Número de triplas: 3
http://noosphera.org/ontology#O2 -> http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> http://noosphera.org/ontology#Object
http://noosphera.org/ontology#O1 -> http://noosphera.org/ontology#relatesTo -> http://noosphera.org/ontology#O2
http://noosphera.org/ontology#O1 -> http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> http://noosphera.org/ontology#Object


#Transformação

In [4]:
estrutura = {
    "nodes": ["O1", "O2"],
    "relations": [
        ("O1", "relatesTo", "O2")
    ]
}

print(estrutura)

{'nodes': ['O1', 'O2'], 'relations': [('O1', 'relatesTo', 'O2')]}


In [5]:
grafo2 = Graph()

for node in estrutura["nodes"]:
    grafo2.add((
        noo[node],
        RDF.type,
        noo.Object
    ))

for sujeito, relacao, objeto in estrutura["relations"]:
    grafo2.add((
        noo[sujeito],
        noo[relacao],
        noo[objeto]
    ))

In [6]:
print("Grafo original:", len(grafo))
print("Grafo reconstruído:", len(grafo2))

print("\nEstruturas iguais?", set(grafo) == set(grafo2))

Grafo original: 3
Grafo reconstruído: 3

Estruturas iguais? True


#Teste 2 — Invariantes da estrutura

##2.1 — **Identidade**

In [7]:
identidade_original = {"O1", "O2"}

identidade_transformada = {
    "O1",
    "O2"
}

assert identidade_original == identidade_transformada
print("✓ Identidade preservada")

✓ Identidade preservada


In [8]:
#mutação proposital

identidade_transformada = {
    "O1",
    "O3"
}

assert identidade_original == identidade_transformada

AssertionError: 

##Teste 2.2 — Cardinalidade

In [9]:
estrutura = {
    "nodes": ["O1", "O2"],
    "relations": [
        ("O1", "relatesTo", "O2")
    ]
}

assert len(estrutura["nodes"]) == 2

print("✓ Cardinalidade preservada")

✓ Cardinalidade preservada


In [10]:
estrutura_ruim = {
    "nodes": ["O1"],
    "relations": []
}

assert len(estrutura["nodes"]) == len(estrutura_ruim["nodes"])

AssertionError: 

## Teste 2.3 — Direcionalidade

In [11]:
relacao_original = ("O1", "relatesTo", "O2")
relacao_transformada = ("O1", "relatesTo", "O2")

assert relacao_original == relacao_transformada

print("✓ Direcionalidade preservada")

✓ Direcionalidade preservada


In [12]:
relacao_ruim = ("O2", "relatesTo", "O1")

assert relacao_original == relacao_ruim

AssertionError: 

##Teste 2.4 — Tipo da relação

In [13]:
("O1", "O2")

('O1', 'O2')

In [14]:
("O1", "relatesTo", "O2")

('O1', 'relatesTo', 'O2')

In [16]:
assert ("O1") == ("O2")

AssertionError: 

O nome relatesTo pertence à estrutura ou é uma propriedade da representação?

##Teste 3 — Isomorfismo

In [17]:
#Representação A:

A = {
    "nodes": {"O1", "O2"},
    "edges": {
        ("O1", "R", "O2")
    }
}

In [18]:
#Representação B:

B = {
    "nodes": {"x", "y"},
    "edges": {
        ("x", "R", "y")
    }
}

In [ ]:
A["nodes"] != B["nodes"]

True

In [19]:
mapeamento = {
    "O1": "x",
    "O2": "y"
}

In [20]:
A_transformado = {
    (mapeamento[s], p, mapeamento[o])
    for s, p, o in A["edges"]
}

print(A_transformado)

{('x', 'R', 'y')}


In [21]:
assert A_transformado == B["edges"]

print("✓ Estruturas isomorfas")

✓ Estruturas isomorfas


#Teste 4 — Trocar completamente o substrato

In [22]:
estrutura = {
    "nodes": ["A", "B"],
    "edges": [
        ("A", "R", "B")
    ]
}

In [23]:
#Representação 1 — tupla

representacao_tupla = (
    ("A", "R", "B"),
)

In [24]:
#Representação 2 — dicionário

representacao_dict = {
    "source": "A",
    "relation": "R",
    "target": "B"
}

In [25]:
#Representação 3 — objeto

class Relation:
    def __init__(self, source, relation, target):
        self.source = source
        self.relation = relation
        self.target = target

representacao_objeto = Relation("A", "R", "B")

In [26]:
def extrair_estrutura(rep):
    if isinstance(rep, tuple):
        return rep[0]

    if isinstance(rep, dict):
        return (
            rep["source"],
            rep["relation"],
            rep["target"]
        )

    if isinstance(rep, Relation):
        return (
            rep.source,
            rep.relation,
            rep.target
        )

In [27]:
assert extrair_estrutura(representacao_tupla) == ("A", "R", "B")

assert extrair_estrutura(representacao_dict) == ("A", "R", "B")

assert extrair_estrutura(representacao_objeto) == ("A", "R", "B")

print("✓ Três representações preservam a mesma estrutura")

✓ Três representações preservam a mesma estrutura


#Experimento 5 — Quando a estrutura muda, mas permanece a mesma?

In [28]:
A = {
    "nodes": {"A", "B", "C"},
    "edges": {
        ("A", "R", "B"),
        ("A", "R", "C")
    }
}

In [29]:
B = {
    "nodes": {"X", "Y", "Z"},
    "edges": {
        ("X", "R", "Y"),
        ("X", "R", "Z")
    }
}

In [30]:
C = {
    "nodes": {"X", "Y", "Z"},
    "edges": {
        ("X", "R", "Y"),
        ("X", "R", "Z"),
        ("Y", "R", "Z")
    }
}

#A estrutura A e a estrutura C são a mesma estrutura em algum sentido relevante?
#O que exatamente deixou de ser preservado quando passamos de A para C?

Investigar usando apenas:

- número de nós;
- número de relações;
- direção das relações;
- quais nós estão relacionados;
- posição estrutural de cada nó.

Um pequeno desafio adicional

Escreva uma função:

```python

def comparar(A, B):
    ...
```

que não compare os nomes dos nós.

Ela deve tentar responder:

```
mesma estrutura
      ou
estrutura diferente
```

Por exemplo:

```
comparar(A, B)
```

deveria encontrar uma equivalência estrutural.

```
comparar(A, C)
```

deveria encontrar alguma diferença.


In [37]:
def _get_canonical_form(graph_data):
    """
    Cria uma representação canônica de um grafo onde os nomes dos nós são ignorados.
    Os nós são mapeados para índices ordenados e as arestas são reescritas
    usando esses índices.
    """
    nodes = graph_data["nodes"]
    edges = graph_data["edges"]

    if not nodes: # Lida com o caso de grafo vazio para evitar erro no sorted()
        return frozenset()

    # Cria um mapeamento ordenado de nomes de nós para índices
    sorted_nodes = sorted(list(nodes))
    node_to_idx = {node: i for i, node in enumerate(sorted_nodes)}

    # Transforma as arestas usando os índices canônicos
    canonical_edges = frozenset({(node_to_idx[s], p, node_to_idx[o])
                                 for s, p, o in edges})
    return canonical_edges

def comparar(A, B):
    """
    Compara duas estruturas de grafo (A e B) para equivalência estrutural,
    ignorando os nomes dos nós.
    """
    # Verifica primeiro as cardinalidades (número de nós e arestas)
    # Essas são condições necessárias para isomorfismo.
    if len(A["nodes"]) != len(B["nodes"]):
        return False
    if len(A["edges"]) != len(B["edges"]):
        return False

    # Gera as formas canônicas para A e B
    canonical_A = _get_canonical_form(A)
    canonical_B = _get_canonical_form(B)

    # Compara as formas canônicas
    return canonical_A == canonical_B

# Testar a função comparar com A, B e C
comparar_A_B_resultado = "mesma estrutura" if comparar(A, B) else "estrutura diferente"
comparar_A_C_resultado = "mesma estrutura" if comparar(A, C) else "estrutura diferente"

print(f"Comparar A e B: {comparar_A_B_resultado}")
print(f"Comparar A e C: {comparar_A_C_resultado}")

Comparar A e B: mesma estrutura
Comparar A e C: estrutura diferente


Com base nas análises e na função `comparar` que criamos, a resposta é:

O nome `relatesTo` (ou qualquer outro nome de relação, como 'R') **pertence à estrutura** do grafo.

Veja por que:

1.  **Isomorfismo de Grafos**: Nossa função `comparar` foi projetada para ignorar os *nomes dos nós* (mapeando-os para índices canônicos), mas ela **preserva os nomes das relações** (`p` no tupla `(s, p, o)`). Isso significa que, para a função considerar duas estruturas iguais, as relações entre os nós (após o mapeamento dos nós) precisam ter o *mesmo tipo* de relação.
2.  **Definição da Conectividade**: O nome da relação define a natureza do vínculo entre dois nós. Se alterarmos 'relatesTo' para 'hasA' (por exemplo), mesmo que entre os mesmos nós, estamos descrevendo uma conectividade diferente, e isso muda a estrutura semântica e topológica do grafo. As arestas em um grafo não são apenas existências; elas têm *propriedades*, e o tipo da relação é uma delas.

Em resumo, enquanto os identificadores dos nós podem ser permutados sem alterar a estrutura fundamental de conectividade (se o grafo for isomórfico), o nome da relação é um componente intrínseco que define essa conectividade e, portanto, é parte da estrutura.

## Teste com Grafos Mais Complexos

In [38]:
# Grafo D: Um ciclo simples de 4 nós
D = {
    "nodes": {"N1", "N2", "N3", "N4"},
    "edges": {
        ("N1", "conecta", "N2"),
        ("N2", "conecta", "N3"),
        ("N3", "conecta", "N4"),
        ("N4", "conecta", "N1")
    }
}

# Grafo E: Isomorfo a D, com nomes de nós diferentes
E = {
    "nodes": {"A", "B", "C", "D"},
    "edges": {
        ("A", "conecta", "B"),
        ("B", "conecta", "C"),
        ("C", "conecta", "D"),
        ("D", "conecta", "A")
    }
}

# Grafo F: Não isomorfo a D ou E (tem uma aresta a mais)
F = {
    "nodes": {"V1", "V2", "V3", "V4"},
    "edges": {
        ("V1", "conecta", "V2"),
        ("V2", "conecta", "V3"),
        ("V3", "conecta", "V4"),
        ("V4", "conecta", "V1"),
        ("V1", "conecta", "V3") # Aresta adicional
    }
}

# Grafo G: Também não isomorfo a D ou E (tem um nó a mais)
G = {
    "nodes": {"X1", "X2", "X3", "X4", "X5"},
    "edges": {
        ("X1", "conecta", "X2"),
        ("X2", "conecta", "X3"),
        ("X3", "conecta", "X4"),
        ("X4", "conecta", "X5"),
        ("X5", "conecta", "X1")
    }
}

In [39]:
# Testar a função comparar com os novos grafos
print(f"Comparar D e E: {'mesma estrutura' if comparar(D, E) else 'estrutura diferente'}")
print(f"Comparar D e F: {'mesma estrutura' if comparar(D, F) else 'estrutura diferente'}")
print(f"Comparar D e G: {'mesma estrutura' if comparar(D, G) else 'estrutura diferente'}")

Comparar D e E: mesma estrutura
Comparar D e F: estrutura diferente
Comparar D e G: estrutura diferente


Ela nos permitiu identificar:

Que grafos com diferentes nomes de nós, mas a mesma estrutura de conexão (como A e B, ou D e E), são considerados equivalentes.
Que alterações no número de nós, arestas ou na forma como eles se conectam (como A e C, ou D e F, D e G) resultam em estruturas diferentes.

### Demonstração: `comparar` com Múltiplos Tipos de Relação

Vamos criar dois grafos que são idênticos em termos de nós e número de arestas, mas diferem nos tipos de relação.

In [40]:
# Grafo H: Tem duas relações de tipos diferentes
H = {
    "nodes": {"N1", "N2", "N3"},
    "edges": {
        ("N1", "tem_parte", "N2"),
        ("N2", "depende_de", "N3")
    }
}

# Grafo I: Idêntico ao H, mas com nomes de nós diferentes
I = {
    "nodes": {"A", "B", "C"},
    "edges": {
        ("A", "tem_parte", "B"),
        ("B", "depende_de", "C")
    }
}

# Grafo J: Estruturalmente parecido, mas com um tipo de relação diferente
J = {
    "nodes": {"X", "Y", "Z"},
    "edges": {
        ("X", "tem_parte", "Y"),
        ("Y", "conecta", "Z") # 'conecta' ao invés de 'depende_de'
    }
}

print(f"Comparar H e I: {'mesma estrutura' if comparar(H, I) else 'estrutura diferente'}")
print(f"Comparar H e J: {'mesma estrutura' if comparar(H, J) else 'estrutura diferente'}")

Comparar H e I: mesma estrutura
Comparar H e J: estrutura diferente


Duas estruturas de representação do conhecimento distintas ($A$ e $B$) são isomórficas se, e somente se, houver um mapeamento biunívoco entre seus elementos que **preserve a integridade das suas relações internas**.
*   A identidade do significado não reside nos nós (os dados isolados), mas na **topologia das relações** que esses nós estabelecem.
*   Se mudarmos o protocolo de rede, o formato do banco de dados (de Relacional para Grafos) ou o modelo de linguagem, mas mantivermos o isomorfismo das relações lógicas, a identidade histórica do significado é preservada.

Próximo passo: Descobrir se a matemática que estou procurando é realmente Teoria dos Grafos, ou se preciso subir mais um nível de abstração.